In [1]:
# pip install langchain langchain-community langchain-chroma langchain-huggingface chromadb sentence-transformers

In [ ]:
# Import json library for loading menu data
import json

# Import Document class to create structured documents for retrieval
from langchain_core.documents import Document
# Import PromptTemplate to create customizable prompts for the LLM
from langchain_core.prompts import PromptTemplate
# Import Chroma vector store for semantic search on embedded documents
from langchain_chroma import Chroma
# Import HuggingFace embeddings and pipeline for generating embeddings and text
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
# Import RetrievalQA chain to combine retrieval with question answering
from langchain.chains import RetrievalQA

# Import warnings to suppress non-critical warning messages
import warnings
# Ignore all warnings for cleaner output
warnings.filterwarnings("ignore")

C:\Users\FGstore\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Open the menu.json file in read mode with UTF-8 encoding
with open("menu.json", "r", encoding="utf-8") as f:
    # Load the JSON data into a Python dictionary
    menu = json.load(f)

In [ ]:
# Initialize empty list to store Document objects
docs = []

# Create documents for individual items
# Iterate through each category in the menu
for category, foods in menu.items():
    # For each food item in the category
    for food in foods:
        # Extract item name from food dictionary
        name = food.get("name", "")
        # Extract item description from food dictionary
        description = food.get("description", "")
        # Extract item price from food dictionary
        price = food.get("price", "")

        # Format content as readable text with category, name, description, and price
        content = f"""
Category: {category}
Name: {name}
Description: {description}
Price: {price}
"""

        # Create a Document object with formatted content and metadata
        docs.append(
            Document(
                page_content=content,
                # Store metadata for filtering and identification
                metadata={
                    "category": category,
                    "name": name,
                    "price": price,
                    "type": "item"
                }
            )
        )

# Create category-level documents for better retrieval
# Iterate through each category to create overview documents
for category, foods in menu.items():
    # Create a comma-separated list of all items in the category with prices
    items_list = ", ".join([f"{food['name']} (${food['price']})" for food in foods])
    # Create category overview content with all items and descriptions
    category_content = f"""
Category: {category}
Items in {category}: {items_list}

Available {category.lower()}:
"""
    # Add detailed information for each item in the category
    for food in foods:
        category_content += f"\n- {food['name']}: {food['description']} - ${food['price']}"
    
    # Create a category-level Document for better semantic matching
    docs.append(
        Document(
            page_content=category_content,
            # Store metadata for category-level filtering
            metadata={
                "category": category,
                "type": "category",
                "item_count": len(foods)
            }
        )
    )

In [ ]:
# Create embeddings using HuggingFace's sentence-transformers model
embeddings = HuggingFaceEmbeddings(
    # Use a lightweight model optimized for semantic similarity (L6 = lightweight, MiniLM = minimized)
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
# Import os module for path operations
import os
# Import shutil for directory operations like deletion
import shutil

# Check if the chroma_menu_db directory already exists
if os.path.exists("./chroma_menu_db"):
    # Delete the existing directory to ensure a fresh database with all updates
    shutil.rmtree("./chroma_menu_db")  # Delete old database

# Create a new Chroma vector store from the documents
vectorstore = Chroma.from_documents(
    # Use the documents created from the menu JSON
    documents=docs,
    # Use the embeddings model to convert text to vectors
    embedding=embeddings,
    # Name this collection for organization
    collection_name="menu_collection",
    # Persist the vectorstore to disk for later retrieval
    persist_directory="./chroma_menu_db"
)

In [ ]:
# Create a retriever from the vector store for semantic search
retriever = vectorstore.as_retriever(
    # Use similarity-based search to find relevant documents
    search_type="similarity",
    # Return top 10 most similar documents to ensure category docs are included
    search_kwargs={"k": 10}  # Get more documents to ensure category docs are included
)

In [ ]:
# Create a language model from HuggingFace for text generation
llm = HuggingFacePipeline.from_model_id(
    # Use Qwen's lightweight model (0.5B parameters) optimized for instruction-following
    model_id="Qwen/Qwen2.5-0.5B-Instruct",
    # Specify the task as text generation
    task="text-generation",
    # Configure pipeline parameters
    pipeline_kwargs={
        # Limit response to 200 new tokens
        "max_new_tokens": 200,
        # Lower temperature (0.2) for more deterministic/focused responses
        "temperature": 0.2,
        # Return only the generated text, not the input prompt
        "return_full_text": False
    }
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Device set to use cuda:0


In [ ]:
# Create a prompt template for the assistant
prompt = PromptTemplate(
    # Define the prompt structure with instructions and placeholders
    template="""
You are a helpful restaurant menu assistant.

Use ONLY the menu context below to answer questions.

If the user asks about items in a category (e.g., "what drinks do you have"), 
provide a comprehensive list of all available items in that category with their prices and descriptions.

If the user asks about a specific item, provide details about that item.

If the answer is not in the menu, say:
"I don't have this item in the menu."

Menu context:
{context}

Question:
{question}

Answer:
""",
    # List the variable names that will be inserted into the template
    input_variables=["context", "question"]
)

In [ ]:
# Create a RetrievalQA chain that combines retrieval with question answering
qa_chain = RetrievalQA.from_chain_type(
    # Use the language model for generating answers
    llm=llm,
    # Use the retriever to fetch relevant menu documents
    retriever=retriever,
    # Use "stuff" chain type to directly insert documents into prompt
    chain_type="stuff",
    # Pass the custom prompt template to the chain
    chain_type_kwargs={"prompt": prompt},
    # Return source documents along with the answer for transparency
    return_source_documents=True
)

In [ ]:
# Start an interactive chat loop for menu queries
while True:
    # Prompt user for input
    question = input("\nAsk about the menu: ")
    # Display the user's question
    print(f"\nquestion: {question}")
    # Check if user wants to exit the conversation
    if question.lower() in ["exit", "quit", "stop"]:
        # Break out of the loop to end the program
        break

    # Send the question to the QA chain and get results
    result = qa_chain.invoke({"query": question})

    # Display the retrieved documents for transparency
    print("\nDocuments Retrieved:")
    # Check if any documents were retrieved
    if result["source_documents"]:
        # Loop through each retrieved document with numbering
        for i, doc in enumerate(result["source_documents"], 1):
            # Print the metadata (category, type, etc.) of each document
            print(f"  {i}. {doc.metadata}")
    else:
        # Inform user if no relevant documents were found
        print("No documents found!")

    # Display the bot's generated answer
    print("\nBot:", result["result"])

question: what drinks do you have

Documents Retrieved:
  1. {'price': 35, 'type': 'item', 'category': 'Drinks', 'name': 'Sprite'}
  2. {'item_count': 8, 'type': 'category', 'category': 'Drinks'}
  3. {'type': 'item', 'price': 80, 'name': 'Latte', 'category': 'Drinks'}
  4. {'type': 'item', 'name': 'Coca Cola', 'price': 35, 'category': 'Drinks'}
  5. {'type': 'item', 'price': 65, 'name': 'Iced Coffee', 'category': 'Drinks'}
  6. {'name': 'Cappuccino', 'category': 'Drinks', 'price': 75, 'type': 'item'}
  7. {'name': 'Mango Juice', 'price': 60, 'type': 'item', 'category': 'Drinks'}
  8. {'name': 'Orange Juice', 'price': 50, 'category': 'Drinks', 'type': 'item'}
  9. {'name': 'Mineral Water', 'type': 'item', 'price': 20, 'category': 'Drinks'}
  10. {'category': 'Desserts', 'price': 140, 'type': 'item', 'name': 'Tiramisu'}

Bot: Sprite, Coca Cola, Iced Coffee, Cappuccino, Mango Juice, Orange Juice. 

I don't have this item in the menu. 

Category: Drinks
Name: Sprite
Description: Lemon fla